# 06 - Geo Clustering

In [ ]:
# Cell 1: Setup — reuse train/test from 05
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Assumes train_df, test_df, X_train_new, X_test_new, y_train, y_test
# already exist from 05_feature_engineering.ipynb (baseline + bed/bath/age features)

In [ ]:
# Cell 2: Elbow method to pick k
# Uses Latitude/Longitude — adjust column names if yours differ
coords_train = train_df[['Latitude', 'Longitude']].dropna()

inertias = []
k_range = range(5, 51, 5)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(coords_train)
    inertias.append(km.inertia_)

import matplotlib.pyplot as plt
plt.plot(list(k_range), inertias, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method — Geo Clusters')
plt.show()

In [ ]:
# Cell 3: Fit KMeans with chosen k
# Pick k from the elbow plot above — start with a reasonable guess like 20
K = 20

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
kmeans.fit(train_df[['Latitude', 'Longitude']])

train_df['LocationCluster'] = kmeans.predict(train_df[['Latitude', 'Longitude']])
test_df['LocationCluster'] = kmeans.predict(test_df[['Latitude', 'Longitude']])

print(train_df['LocationCluster'].value_counts().sort_index())

In [ ]:
# Cell 4: Build new feature set — old + geo cluster (one-hot)
cluster_dummies_train = pd.get_dummies(train_df['LocationCluster'], prefix='GeoCluster')
cluster_dummies_test = pd.get_dummies(test_df['LocationCluster'], prefix='GeoCluster')

X_train_geo = pd.concat([X_train_new.reset_index(drop=True), cluster_dummies_train.reset_index(drop=True)], axis=1)
X_test_geo = pd.concat([X_test_new.reset_index(drop=True), cluster_dummies_test.reset_index(drop=True)], axis=1)

# align in case some clusters missing from test
X_train_geo, X_test_geo = X_train_geo.align(X_test_geo, join='left', axis=1, fill_value=0)

print('Feature count with geo clusters:', X_train_geo.shape[1])

In [ ]:
# Cell 5: Train + evaluate RF with geo clusters
rf_geo = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_geo.fit(X_train_geo, y_train)

pred_log = rf_geo.predict(X_test_geo)
pred_price = np.exp(pred_log)
actual_price = np.exp(y_test)

r2_geo = r2_score(y_test, pred_log)
mape_geo = mean_absolute_percentage_error(actual_price, pred_price)
mdape_geo = np.median(np.abs((actual_price - pred_price) / actual_price))

print(f"RF with geo clusters — R²: {r2_geo:.4f}, MAPE: {mape_geo:.4f}, MdAPE: {mdape_geo:.4f}")

In [ ]:
# Cell 6: Comparison table
comparison = pd.DataFrame({
    'Feature Set': ['RF baseline', 'RF + BedBathRatio/PropertyAge', 'RF + Geo Clusters'],
    'R2': [0.8797, r2_new, r2_geo],
    'MAPE': [0.1819, mape_new, mape_geo],
    'MdAPE': [0.1032, mdape_new, mdape_geo]
})
print(comparison)